# Social Media Engagement Analysis

**A data analytics project using Python to explore social media post performance.**

This notebook examines engagement patterns across platforms, post formats, and publishing times. It uses descriptive analysis and visualizations to explore how likes, comments, shares, impressions, and reach vary across the available records.

> **Data note:** Confirm the original dataset publisher, licence, and whether the records are real, anonymized, or synthetic before making claims about real-world audience behaviour. This notebook does not establish that the data comes directly from social media platforms.

## 1. Objectives

- Prepare and inspect the social media post dataset.
- Calculate total interactions and an impressions-based engagement rate.
- Compare engagement across post formats and platforms.
- Explore engagement by posting hour and month.
- Use static and interactive visualizations to communicate patterns.

## 2. Tools and libraries

Python, Pandas, NumPy, Matplotlib, Seaborn, and Plotly.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from pathlib import Path

sns.set_theme(style="whitegrid")
plt.rcParams["font.size"] = 11
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

print("Libraries imported successfully!")

## 3. Load the dataset

Place the CSV in a folder named `dataset` as `Social_Media_Engagement.csv`. In Google Colab, the upload prompt is used if the file is not found. When running locally, the notebook also checks the current directory.

In [ ]:
candidate_paths = [
    Path("dataset/Social_Media_Engagement.csv"),
    Path("Social_Media_Engagement.csv"),
]

file_path = next((p for p in candidate_paths if p.exists()), None)

if file_path is None:
    try:
        from google.colab import files
        print("Upload Social_Media_Engagement.csv")
        uploaded = files.upload()
        file_path = Path(next(iter(uploaded)))
    except ImportError:
        raise FileNotFoundError(
            "Dataset not found. Place Social_Media_Engagement.csv "
            "inside the dataset/ folder or in the notebook directory."
        )

df_raw = pd.read_csv(file_path)
df = df_raw.copy()

# Standardize column names
df.columns = (
    df.columns.str.strip()
              .str.lower()
              .str.replace(r"[^a-z0-9]+", "_", regex=True)
              .str.strip("_")
)

print(f"Raw dataset shape: {df_raw.shape}")
print(f"Columns: {df.columns.tolist()}")
display(df.head())

## 4. Data cleaning and quality checks

The notebook standardizes column names, converts timestamps and metric fields, and removes exact duplicate rows. Missing values are reported rather than automatically filled with zero: a missing interaction count or impression count is not necessarily equivalent to a true zero.

In [ ]:
if "post_timestamp" in df.columns:
    df["post_timestamp"] = pd.to_datetime(df["post_timestamp"], errors="coerce")

numeric_cols = [
    "likes", "comments", "shares", "impressions", "reach", "engagement_rate"
]
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

before_dedup = len(df)
df = df.drop_duplicates().copy()
duplicates_removed = before_dedup - len(df)

# Report missingness after type conversion and duplicate removal
missing_report = (
    df.isna().sum()
      .rename("missing_values")
      .to_frame()
)
missing_report["missing_percent"] = (
    missing_report["missing_values"] / len(df) * 100
).round(2)

print(f"Rows after removing exact duplicates: {len(df):,}")
print(f"Exact duplicate rows removed: {duplicates_removed:,}")
display(missing_report.sort_values("missing_percent", ascending=False))

# Flag impossible negative values in count/volume fields; do not silently alter them
count_cols = [c for c in ["likes", "comments", "shares", "impressions", "reach"] if c in df.columns]
negative_report = {c: int((df[c] < 0).sum()) for c in count_cols}
print("Negative values in count fields:", negative_report)

## 5. Feature engineering and metric definitions

**Total engagement** is defined here as likes + comments + shares.

**Engagement rate (impressions-based)** is defined as:

\[
\text{Engagement Rate (\%)} =
\frac{\text{Likes + Comments + Shares}}{\text{Impressions}} \times 100
\]

Rows with missing engagement components or non-positive/missing impressions are not assigned a calculated rate. This is a post-level rate; it should not be confused with a reach-based or follower-based rate. The dataset's pre-existing `engagement_rate` column is retained separately and is not assumed to use the same formula.

In [ ]:
engagement_parts = [c for c in ["likes", "comments", "shares"] if c in df.columns]
if len(engagement_parts) == 3:
    df["total_engagement"] = df[engagement_parts].sum(axis=1, min_count=3)
else:
    raise ValueError("The dataset must contain likes, comments, and shares columns.")

if "post_timestamp" in df.columns:
    df["post_date"] = df["post_timestamp"].dt.date
    df["post_month"] = df["post_timestamp"].dt.to_period("M").astype(str)
    df["post_hour"] = df["post_timestamp"].dt.hour
else:
    raise ValueError("The dataset must contain a post_timestamp column.")

if "impressions" not in df.columns:
    raise ValueError("The dataset must contain an impressions column.")

df["engagement_rate_pct"] = np.where(
    df["impressions"] > 0,
    (df["total_engagement"] / df["impressions"]) * 100,
    np.nan
)

print(f"Cleaned dataset shape: {df.shape}")
display(df[["likes", "comments", "shares", "impressions",
            "total_engagement", "engagement_rate_pct"]].head())

## 6. Exploratory data analysis and visualizations

The four core charts compare engagement rate by post format, average engagement by posting hour, average impressions and reach by platform, and average monthly engagement.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

if {"post_type", "engagement_rate_pct"}.issubset(df.columns):
    sns.barplot(
        data=df, x="post_type", y="engagement_rate_pct",
        hue="post_type", palette="viridis", ax=axes[0, 0],
        errorbar=None, legend=False
    )
    axes[0, 0].set_title("1. Average Engagement Rate by Post Type")
    axes[0, 0].set_xlabel("Post type")
    axes[0, 0].set_ylabel("Engagement rate (%)")

hourly = df.groupby("post_hour", as_index=False)["total_engagement"].mean()
sns.lineplot(data=hourly, x="post_hour", y="total_engagement",
             marker="o", ax=axes[0, 1])
axes[0, 1].set_title("2. Average Engagement by Posting Hour")
axes[0, 1].set_xlabel("Hour of day (24-hour clock)")
axes[0, 1].set_ylabel("Average total engagement")
axes[0, 1].set_xticks(range(0, 24, 2))

if {"platform", "impressions", "reach"}.issubset(df.columns):
    platform_summary = (
        df.groupby("platform")[["impressions", "reach"]]
          .mean().reset_index()
          .melt(id_vars="platform", var_name="metric", value_name="average")
    )
    sns.barplot(data=platform_summary, x="platform", y="average",
                hue="metric", ax=axes[1, 0])
    axes[1, 0].set_title("3. Average Impressions and Reach by Platform")
    axes[1, 0].set_xlabel("Platform")
    axes[1, 0].set_ylabel("Average count")

monthly = (
    df.groupby("post_month", as_index=False)["total_engagement"]
      .mean()
)
sns.lineplot(data=monthly, x="post_month", y="total_engagement",
             marker="s", ax=axes[1, 1])
axes[1, 1].set_title("4. Average Monthly Engagement")
axes[1, 1].set_xlabel("Post month")
axes[1, 1].set_ylabel("Average total engagement")
axes[1, 1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

## 7. Additional interactive visualizations

The following Plotly charts provide an interactive view of post-format distribution, monthly engagement by format, and impressions versus total engagement over time.

In [ ]:
# Interactive donut chart: distribution of post types
if "post_type" in df.columns:
    type_counts = df["post_type"].value_counts().reset_index()
    type_counts.columns = ["post_type", "count"]
    fig_donut = px.pie(
        type_counts, values="count", names="post_type", hole=0.5,
        title="Distribution of Post Types",
        color_discrete_sequence=px.colors.qualitative.Pastel
    )
    fig_donut.update_traces(textposition="inside", textinfo="percent+label")
    fig_donut.show()

# Animated bar chart: average monthly engagement by post type
monthly_type = (
    df.groupby(["post_month", "post_type"], as_index=False)["total_engagement"]
      .mean().sort_values("post_month")
)
fig_anim_bar = px.bar(
    monthly_type, x="post_type", y="total_engagement",
    color="post_type", animation_frame="post_month",
    range_y=[0, monthly_type["total_engagement"].max() * 1.15],
    title="Average Monthly Engagement by Post Type",
    labels={"total_engagement": "Average total engagement", "post_type": "Post format"},
    color_discrete_sequence=px.colors.qualitative.Vivid
)
fig_anim_bar.update_layout(transition_duration=500)
fig_anim_bar.show()

# Animated scatter: impressions versus engagement, with reach as bubble size
df_anim = df.dropna(subset=["impressions", "total_engagement", "reach", "platform"]).copy()
df_anim = df_anim[(df_anim["impressions"] > 0) & (df_anim["reach"] > 0)]
df_anim["animation_month"] = df_anim["post_timestamp"].dt.strftime("%Y-%m")
fig_anim_scatter = px.scatter(
    df_anim, x="impressions", y="total_engagement",
    animation_frame="animation_month", size="reach", color="platform",
    hover_name="post_type", log_x=True, size_max=35,
    title="Impressions versus Total Engagement Over Time",
    labels={"impressions": "Impressions (log scale)",
            "total_engagement": "Total engagement"}
)
fig_anim_scatter.show()

## 8. Summary tables for interpretation

Use these summaries to support written observations. The notebook intentionally does not hard-code conclusions: the findings should be written after reviewing the actual outputs.

In [ ]:
if "post_type" in df.columns:
    display(
        df.groupby("post_type")
          .agg(posts=("post_type", "size"),
               avg_engagement=("total_engagement", "mean"),
               avg_engagement_rate_pct=("engagement_rate_pct", "mean"),
               avg_reach=("reach", "mean"))
          .sort_values("avg_engagement_rate_pct", ascending=False)
          .round(2)
    )

if "platform" in df.columns:
    display(
        df.groupby("platform")
          .agg(posts=("platform", "size"),
               avg_engagement=("total_engagement", "mean"),
               avg_impressions=("impressions", "mean"),
               avg_reach=("reach", "mean"))
          .round(2)
    )

## 9. Limitations

- The results describe the supplied dataset only; they should not be generalized to all social media users or platforms.
- Engagement rate depends on the denominator. This notebook uses impressions, not reach or followers.
- A high average may be influenced by outliers or unequal numbers of posts across groups.
- Posting-time patterns are descriptive associations, not proof that a particular hour causes higher engagement.
- Dataset provenance and licence must be verified and cited in the repository before public redistribution.

## 10. Conclusion

This analysis provides a descriptive view of social media post engagement across formats, platforms, and time. Interpret the summary tables and charts together, and base final recommendations on the observed values rather than assumptions.